In [7]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [2]:
import os
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Retrieve the API key
google_api_key = os.getenv("GOOGLE_API_KEY")

# Verify it was loaded successfully (masks key for security)
if google_api_key:
    print(f"Loaded successfully: {google_api_key[:6]}...{google_api_key[-4:]}")
else:
    print("GOOGLE_API_KEY not found in .env")


Loaded successfully: AIzaSy...TC0w


In [3]:
# Configure Retry Options
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)

### Why Multi-Agent Systems? + Your First Multi-Agent[¶](https://www.kaggle.com/code/kaggle5daysofai/day-1b-agent-architectures#%F0%9F%A4%94-Section-2:-Why-Multi-Agent-Systems?-+-Your-First-Multi-Agent)

**The Problem: The "Do-It-All" Agent**

Single agents can do a lot. But what happens when the task gets complex? A single "monolithic" agent that tries to do research, writing, editing, and fact-checking all at once becomes a problem. Its instruction prompt gets long and confusing. It's hard to debug (which part failed?), difficult to maintain, and often produces unreliable results.

**The Solution: A Team of Specialists**

Instead of one "do-it-all" agent, we can build a **multi-agent system**. This is a team of simple, specialized agents that collaborate, just like a real-world team. Each agent has one clear job (e.g., one agent *only* does research, another *only* writes). This makes them easier to build, easier to test, and much more powerful and reliable when working together.

To learn more, check out the documentation related to [LLM agents in ADK](https://google.github.io/adk-docs/agents/llm-agents/).

**Architecture: Single Agent vs Multi-Agent Team**

![Multi-agent Team](https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/multi-agent-team.png)

### 2.1 Example: Research & Summarization System

Let's build a system with two specialized agents:

1. **Research Agent** - Searches for information using Google Search
2. **Summarizer Agent** - Creates concise summaries from research finding

You used a single "coordinator" agent to manage the workflow, which is a powerful and flexible pattern.
This is the first Multi-Agent System!

‼️ However, relying on an LLM's instructions to control the order can sometimes be unpredictable. 

In [4]:
# Research Agent: Its job is to use the google_search tool and present findings.
research_agent = Agent(
    name="ResearchAgent",
    model=Gemini(
        model="gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a specialized research agent. Your only job is to use the
    google_search tool to find 2-3 pieces of relevant information on the given topic and present the findings with citations.""",
    tools=[google_search],
    output_key="research_findings",  # The result of this agent will be stored in the session state with this key.
)

print("✅ research_agent created.")

✅ research_agent created.


In [5]:
# Summarizer Agent: Its job is to summarize the text it receives.
summarizer_agent = Agent(
    name="SummarizerAgent",
    model=Gemini(
        model="gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    # The instruction is modified to request a bulleted list for a clear output format.
    instruction="""Read the provided research findings: {research_findings}
Create a concise summary as a bulleted list with 3-5 key points.""",
    output_key="final_summary",
)

print("✅ summarizer_agent created.")

✅ summarizer_agent created.


In [8]:
# Root Coordinator: Orchestrates the workflow by calling the sub-agents as tools.
root_agent = Agent(
    name="ResearchCoordinator",
    model=Gemini(
        model="gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    # This instruction tells the root agent HOW to use its tools (which are the other agents).
    instruction="""You are a research coordinator. Your goal is to answer the user's query by orchestrating a workflow.
1. First, you MUST call the `ResearchAgent` tool to find relevant information on the topic provided by the user.
2. Next, after receiving the research findings, you MUST call the `SummarizerAgent` tool to create a concise summary.
3. Finally, present the final summary clearly to the user as your response.""",
    # We wrap the sub-agents in `AgentTool` to make them callable tools for the root agent.
    tools=[AgentTool(research_agent), AgentTool(summarizer_agent)],
)

print("✅ root_agent created.")

✅ root_agent created.


In [9]:
runner = InMemoryRunner(agent=root_agent)
runner = await runner.run_debug(
    "What is latest advancements in AI, and what is the best way to learn about them?"
)

c:\Users\devas\Documents\Development\AgentDevCourse\.venv\Lib\site-packages\google\adk\models\llm_request.py:306: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


ResearchCoordinator > Here is a comprehensive overview of the latest advancements in Artificial Intelligence and the best ways to get started or level up your knowledge:

---

### **Latest Advancements in AI**
The AI landscape has evolved far beyond basic text chats and simple automations, shifting heavily toward reasoning, autonomy, and deep integration:
1. **Reasoning Models (System 2 Thinking):** Newer AI architectures break complex queries down into multi-step logical paths before answering, mimicking human problem-solving.
2. **Native Multimodality:** State-of-the-art models process text, high-resolution video, audio, and code simultaneously in real-time.
3. **Massive Context Windows:** Models can now process millions of tokens in a single prompt—allowing you to drop entire codebases or multi-volume books directly into a chat.
4. **Edge AI & Small Language Models (SLMs):** Highly optimized lightweight models now run directly on smartphones and local hardware for instant, privacy-f

### Sequential Workflows - The Assembly Line[¶](https://www.kaggle.com/code/kaggle5daysofai/day-1b-agent-architectures#%F0%9F%9A%A5-Section-3:-Sequential-Workflows---The-Assembly-Line)

**The Problem: Unpredictable Order**

The previous multi-agent system worked, but it relied on a **detailed instruction prompt** to force the LLM to run steps in order. This can be unreliable. A complex LLM might decide to skip a step, run them in the wrong order, or get "stuck," making the process unpredictable.

**The Solution: A Fixed Pipeline**

When you need tasks to happen in a **guaranteed, specific order**, you can use a `SequentialAgent`. This agent acts like an assembly line, running each sub-agent in the exact order you list them. The output of one agent automatically becomes the input for the next, creating a predictable and reliable workflow.

**Use Sequential when:** Order matters, you need a linear pipeline, or each step builds on the previous one.

To learn more, check out the documentation related to [sequential agents in ADK](https://google.github.io/adk-docs/agents/workflow-agents/sequential-agents/).

**Architecture: Blog Post Creation Pipeline**

![Sequential Agent](https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/sequential-agent.png)

### 3.1 Example: Blog Post Creation with Sequential Agents

Let's build a system with three specialized agents:

1. **Outline Agent** - Creates a blog outline for a given topic
2. **Writer Agent** - Writes a blog post
3. **Editor Agent** - Edits a blog post draft for clarity and structure


In [11]:
outline_agent = Agent(
    name = "OutLineAgent" ,
    model = Gemini(
        model="gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
        instruction = """ You are an agent to create outline for a blog post. Your job is to:
        1. A catchy headline
        2. An introductory hook
        3. A main section with 3-5 key ideas with 2-3 bullet points for each that can be identified using google search.
        4. A concluding thought that provides an aha moment.
        """,
        tools= [google_search],
        output_key="blog_outline" # The result of this agent will be stored in the session state with this key.
)
print("outline_agent is created")

outline_agent is created


In [12]:
writer_agent = Agent(
    name = "WriterAgent" ,
    model=Gemini(
        model="gemini-3.5-flash-lite" ,
        retry_options=retry_config 
    ),
    instruction= """ 
    Your job is to write the blog post strictly on the {blog_outline}.
    Write a brief 400-500 words blog post with an engaging and informative tone. 
    Make sure to not use any em-dash in the blog post.
    """ ,
    output_key="blog_draft",
)
print("writer_agent is created")

writer_agent is created


In [13]:
editor_agent = Agent(
    name = "EditorAgent" ,
    model = Gemini(
        model = "gemini-3.5-flash-lite" ,
        retry_options=retry_config
    ) ,
    instruction= """ 
    Edit this draft: {blog_draft}.
    Polish the draft by fixing grammatical errors, sentence structure, improving the flow and enhancing clarity.
    """ ,
    output_key='final_blog',
)
print("editor_agent is created")

editor_agent is created


In [15]:
root_agent = SequentialAgent(
    name = "BlogPipeline" ,
    sub_agents=[outline_agent,writer_agent,editor_agent],
)
print("Sequential Agent Created")

Sequential Agent Created


C:\Users\devas\AppData\Local\Temp\ipykernel_31228\3931043205.py:1: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  root_agent = SequentialAgent(


In [16]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Write a blog post on how Hurufism is still practised in contemporary pop culture"
)

OutLineAgent > Here is a comprehensive outline for a blog post exploring how Hurufism manifests in contemporary pop culture:

# Decoding the Cosmos: How 14th-Century Hurufism Secretly Shapes Today’s Pop Culture

### 1. Introductory Hook
Imagine looking at a sprawling street mural, a piece of avant-garde graphic design, or a conceptual music video, feeling a deep sense of modern aesthetic wonder—without realizing you are staring at a coded mystical philosophy born in the 14th century. While medieval Islamic mysticism is rarely talked about over morning coffee, the ancient philosophy of **Hurufism**—the belief that letters and numbers hold the esoteric key to the universe—is quietly alive and well. Far from being confined to dusty historical archives, Hurufism’s obsession with the power of letters continues to pulse beneath the surface of contemporary art, design, and pop culture.

---

### 2. Main Section: Key Ideas and Supporting Points

*   **Idea 1: The *Hurufiyya* Movement and Conte

### Parallel Workflows - Independent Researchers[¶](https://www.kaggle.com/code/kaggle5daysofai/day-1b-agent-architectures#%F0%9F%9B%A3%EF%B8%8F-Section-4:-Parallel-Workflows---Independent-Researchers)

**The Problem: The Bottleneck**

The previous sequential agent is great, but it's an assembly line. Each step must wait for the previous one to finish. What if you have several tasks that are **not dependent** on each other? For example, researching three *different* topics. Running them in sequence would be slow and inefficient, creating a bottleneck where each task waits unnecessarily.

**The Solution: Concurrent Execution**

When you have independent tasks, you can run them all at the same time using a `ParallelAgent`. This agent executes all of its sub-agents concurrently, dramatically speeding up the workflow. Once all parallel tasks are complete, you can then pass their combined results to a final 'aggregator' step.

**Use Parallel when:** Tasks are independent, speed matters, and you can execute concurrently.

To learn more, check out the documentation related to [parallel agents in ADK](https://google.github.io/adk-docs/agents/workflow-agents/parallel-agents/).

**Architecture: Multi-Topic Research**

![Parallel Agent](https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/parallel-agent.png)

### 4.1 Example: Parallel Multi-Topic Research

Let's build a system with four agents:

1. **Tech Researcher** - Researches AI/ML news and trends
2. **Health Researcher** - Researches recent medical news and trends
3. **Finance Researcher** - Researches finance and fintech news and trends
4. **Aggregator Agent** - Combines all research findings into a single summary


In [30]:
tech_researcher = Agent(
    name = "TechResearcher" ,
    model= Gemini(
        model = "gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction= """ 
    Research the latest AI/ML trends. Include 3 key developments, 
    the main companies involved, and the potential impact. Keep the report very concise (100 words).
    """ ,
    output_key='tech_output',
    tools=[google_search],
)

print("tech_researcher Agent is created!")

tech_researcher Agent is created!


In [31]:
health_researcher = Agent(
    name = "HealthResearcher" ,
    model= Gemini(
        model = "gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction= """ 
    Research the latest Health trends and medical advances. Include 3 significant advances, 
    their practical application, and the estimated timeline. Keep the report very concise (100 words).
    """ ,
    output_key='health_output',
    tools=[google_search],
)

print("health_researcher Agent is created!")

health_researcher Agent is created!


In [32]:
finance_researcher = Agent(
    name = "FinanceResearcher" ,
    model= Gemini(
        model = "gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction= """ 
    Research current fintech trends. Include 3 key trends,
    their market implications, and the future outlook. Keep the report concise (100 words).
    """ ,
    output_key='finance_output',
    tools=[google_search],
)

print("finance_researcher Agent is created!")

finance_researcher Agent is created!


In [33]:
# The AggregatorAgent runs *after* the parallel step to synthesize the results.

aggregator_agent = Agent(
    name= "AggregatorAgent" ,
    model = Gemini(
        model = "gemini-3.5-flash-lite" ,
        retry_options=retry_config
    ) ,
    # It uses placeholders to inject the outputs from the parallel agents, which are now in the session state.
    instruction= """ 
    Combine the three research findings into a single executive summary:
    ** Technology Trends: ** 
    {tech_output}

    ** Heath Breakthroughs: **
    {health_output}

    ** Finance Innovations: **
    {finance_output}
    
    Your summary should highlight common themes, surprising connections, and the most important key takeaways from all three reports. 
    The final summary should be around 200 words.
    """ ,
    output_key="executive_summary"
)

print("aggregator_agent is Created!")

aggregator_agent is Created!


In [34]:
# The ParallelAgent runs all its sub-agents simultaneously.
# This design ensures that the research agents run first in parallel, then once all of their research is complete, 
# the aggregator agent brings together all of the research findings into a single report

parallel_research_team = ParallelAgent(
    name = "ParallelResearchTeam",
    sub_agents = [tech_researcher,finance_researcher,health_researcher]

)

root_agent = SequentialAgent(
    name = "ResearchSystem" ,
    sub_agents=[parallel_research_team,aggregator_agent] ,
)

print("Parallel & Sequential Agents created")

Parallel & Sequential Agents created


C:\Users\devas\AppData\Local\Temp\ipykernel_31228\2245213889.py:5: DeprecationWarning: ParallelAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  parallel_research_team = ParallelAgent(
C:\Users\devas\AppData\Local\Temp\ipykernel_31228\2245213889.py:11: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  root_agent = SequentialAgent(


In [35]:
runner = InMemoryRunner(root_agent)
response = await runner.run_debug(
    "Run the Daily Executive briefing on Tech, Health & Finance"
) 

TechResearcher > **Daily Executive Briefing: AI/ML Trends**

**Key Developments:**
1. **Agentic AI in Production:** Systems are evolving from passive text generators into autonomous agents capable of planning, executing multi-step workflows, and utilizing software tools independently.
2. **AI for Scientific Discovery:** Machine learning is heavily accelerating research workflows, driving complex engineering simulations and novel chemical/drug molecule discovery.
3. **Massive Infrastructure Scaling:** Enterprise architectures are prioritizing heavy investments in dedicated power grids, specialized chips, and cybersecurity to support continuous AI operations.

**Main Companies Involved:**
* OpenAI, Microsoft, Google, and enterprise cloud providers scaling autonomous software capabilities.

**Potential Impact:**
Drastically increases operational efficiency and automates complex backend workflows across finance, health, and tech, while demanding major capital updates for energy and digital

### Workflow

Parallel and Sequential agent are deprecated, and are replaced by Workflow. Below is an attempt to replace them with Workflow()

### Syntax Cheat Sheet:

* **Linear / Sequential**: `(START, agent_a, agent_b, agent_c)`
* **Parallel Fan-out**: `(START, (agent_1, agent_2, agent_3))`
* **Parallel + Join / Fan-in**: `(START, (agent_1, agent_2, agent_3), aggregator)`

### Problems Faced:
* mode = 'single turn'
    - Problem: When I try to run the agent using run_debug, I get the error: Value error, The agent 'WriterAgent' has been added to the workflow with mode='chat' following node 'OutLineAgent'. 
    - Why this happens: This is not supported because chat-mode agents rely on conversational history (session events) and cannot consume direct node inputs from preceding nodes.
    - Solution: Please change the agent's mode to 'single_turn'
* JoinNode Issue in Parallel Agent:
    - Problem: Here I was constantly getting the error: "Context variable not found: `health_output` in agent 'AggregatorAgent'."
    - Why this happens: In Workflow, regular agents run as soon as the first predecessor finishes. Because tech_researcher or finance_researcher finishes before health_researcher, aggregator_agent starts immediately while health_output is not yet available in the state.
    - Solution: To wait for all parallel branches before running the aggregator, use JoinNode.

In [ ]:
#### Replacinging Squential Agent with Workflow()

from google.adk.workflow import Workflow, START

outline_agent = Agent(
    name = "OutLineAgent" ,
    model = Gemini(
        model="gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
        instruction = """ You are an agent to create outline for a blog post. Your job is to:
        1. A catchy headline
        2. An introductory hook
        3. A main section with 3-5 key ideas with 2-3 bullet points for each that can be identified using google search.
        4. A concluding thought that provides an aha moment.
        """,
        tools= [google_search],
        output_key="blog_outline" # The result of this agent will be stored in the session state with this key.
)
print("outline_agent is created")

writer_agent = Agent(
    name = "WriterAgent" ,
    model=Gemini(
        model="gemini-3.5-flash-lite" ,
        retry_options=retry_config 
    ),
    instruction= """ 
    Your job is to write the blog post strictly on the {blog_outline}.
    Write a brief 400-500 words blog post with an engaging and informative tone. 
    Make sure to not use any em-dash in the blog post.
    """ ,
    output_key="blog_draft",
    mode="single_turn" , # Newly added, not existing in the earlier version
)
print("writer_agent is created")

editor_agent = Agent(
    name = "EditorAgent" ,
    model = Gemini(
        model = "gemini-3.5-flash-lite" ,
        retry_options=retry_config
    ) ,
    instruction= """ 
    Edit this draft: {blog_draft}.
    Polish the draft by fixing grammatical errors, sentence structure, improving the flow and enhancing clarity.
    """ ,
    output_key='final_blog',
    mode="single_turn" , # Newly added, not existing in the earlier version
)
print("editor_agent is created")

#Replace Sequential Workflow in the Blog Pipeline:
root_agent = Workflow(
    name = "BlogPipeline" ,
    edges=[
        (START,outline_agent,writer_agent,editor_agent)
    ]
)

print("Sequential Workflow Created!")

runner = InMemoryRunner(root_agent)
response = await runner.run_debug(
    "Create a blog post on whether India uses Asymmetrical warfare to counter China in Defence capabilities"
)

outline_agent is created
writer_agent is created
editor_agent is created
Sequential Workflow Created!
OutLineAgent > ### 1. Catchy Headline
**David vs. Dragon: Is India Using Asymmetric Warfare to Outsmart China’s Defense Machine?**

### 2. Introductory Hook
When looking at the sheer metrics of military might, matching China weapon-for-weapon, ship-for-ship, or yuan-for-dollar is a mathematical losing game for India. With Beijing possessing a vastly superior defense budget, advanced industrial capacity, and cutting-edge informationized warfare infrastructure, a symmetrical arms race is practically out of reach. Yet, military strategy isn't just about counting numbers—it’s about turning geography, ingenuity, and unconventional tactics into leverage. As geopolitical tensions persist along the Line of Actual Control (LAC), a critical question arises: Is India quietly leveraging asymmetric warfare to level the playing field against its northern neighbor?

---

### 3. Main Section: Key Idea

In [46]:
#### Replacinging Parallel Agent with Workflow()

from google.adk.workflow import Workflow, START , JoinNode

tech_researcher = Agent(
    name = "TechResearcher" ,
    model= Gemini(
        model = "gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction= """ 
    Research the latest AI/ML trends. Include 3 key developments, 
    the main companies involved, and the potential impact. Keep the report very concise (100 words).
    """ ,
    output_key='tech_output',
    tools=[google_search],
    mode='single_turn',
)

print("tech_researcher Agent is created!")

health_researcher = Agent(
    name = "HealthResearcher" ,
    model= Gemini(
        model = "gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction= """ 
    Research the latest Health trends and medical advances. Include 3 significant advances, 
    their practical application, and the estimated timeline. Keep the report very concise (100 words).
    """ ,
    output_key='health_output',
    tools=[google_search],
    mode='single_turn',
)

print("health_researcher Agent is created!")

finance_researcher = Agent(
    name = "FinanceResearcher" ,
    model= Gemini(
        model = "gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction= """ 
    Research current fintech trends. Include 3 key trends,
    their market implications, and the future outlook. Keep the report concise (100 words).
    """ ,
    output_key='finance_output',
    tools=[google_search],
    mode='single_turn',
)

print("finance_researcher Agent is created!")

# The AggregatorAgent runs *after* the parallel step to synthesize the results.

aggregator_agent = Agent(
    name= "AggregatorAgent" ,
    model = Gemini(
        model = "gemini-3.5-flash-lite" ,
        retry_options=retry_config
    ) ,
    # It uses placeholders to inject the outputs from the parallel agents, which are now in the session state.
    instruction= """ 
    Combine the three research findings into a single executive summary:
    ** Technology Trends: ** 
    {tech_output}

    ** Heath Breakthroughs: **
    {health_output}

    ** Finance Innovations: **
    {finance_output}
    
    Your summary should highlight common themes, surprising connections, and the most important key takeaways from all three reports. 
    The final summary should be around 200 words.
    """ ,
    output_key="executive_summary",
    mode="single_turn",
)

print("aggregator_agent is Created!")

# 1. Create a JoinNode that waits for all parallel agents to finish
join = JoinNode(name="join")

# 2. Place 'join' between the parallel tuple and the aggregator
root_agent = Workflow(
    name = "ResearchSystem" ,
    edges=[
        (START,(tech_researcher,health_researcher,finance_researcher),join, aggregator_agent) 
    ],
)


print("Parallel Research Workflow created")

runner = InMemoryRunner(root_agent)
response = await runner.run_debug("Run a 2025 briefing on tech, health and finance!")

tech_researcher Agent is created!
health_researcher Agent is created!
finance_researcher Agent is created!
aggregator_agent is Created!
Parallel Research Workflow created
TechResearcher > **Tech, Health & Finance Briefing**

*   **Key Developments:** 
    1. **Agentic AI Expansion:** Autonomous AI agents are scaling across tech and finance to automate complex workflows and real-time decision-making.
    2. **Clinical GenAI Integration:** Healthcare systems are heavily deploying generative AI for documentation automation, claims processing, and drug discovery.
    3. **Automated Compliance & Risk:** Financial institutions are leveraging real-time AI modeling to navigate stringent regulations and secure transactions.
*   **Main Companies:** OpenAI, NVIDIA, J.P. Morgan, Microsoft, and specialized health-tech innovators.
*   **Potential Impact:** Drastically reduced administrative overheads, enhanced patient and customer experiences, and optimized risk management.
HealthResearcher > **Tech

## Loop Workflows - The Refinement Cycle[¶](https://www.kaggle.com/code/kaggle5daysofai/day-1b-agent-architectures#%E2%9E%B0-Section-5:-Loop-Workflows---The-Refinement-Cycle)

**The Problem: One-Shot Quality**

All the workflows we've seen so far run from start to finish. The `SequentialAgent` and `ParallelAgent` produce their final output and then stop. This 'one-shot' approach isn't good for tasks that require refinement and quality control. What if the first draft of our story is bad? We have no way to review it and ask for a rewrite.

**The Solution: Iterative Refinement**

When a task needs to be improved through cycles of feedback and revision, you can use a `LoopAgent`. A `LoopAgent` runs a set of sub-agents repeatedly *until a specific condition is met or a maximum number of iterations is reached.* This creates a refinement cycle, allowing the agent system to improve its own work over and over.

**Use Loop when:** Iterative improvement is needed, quality refinement matters, or you need repeated cycles.

To learn more, check out the documentation related to [loop agents in ADK](https://google.github.io/adk-docs/agents/workflow-agents/loop-agents/).

**Architecture: Story Writing & Critique Loop**

![Loop Agent](https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/loop-agent.png)

### 5.1 Example: Iterative Story Refinement

Let's build a system with two agents:

1. **Writer Agent** - Writes a draft of a short story
2. **Critic Agent** - Reviews and critiques the short story to suggest improvements

Now, we need a way for the loop to actually stop based on the critic's feedback. The LoopAgent itself doesn't automatically know that "APPROVED" means "stop."

We need an agent to give it an explicit signal to terminate the loop.

We do this in two parts:

- **Exit Loop** - A simple Python function that the LoopAgent understands as an "exit" signal.
- **Refiner Agent** -  that can call that exit function when the right condition is met.
    This agent is the "brain" of the loop. It reads the {critique} from the CriticAgent and decides whether to (1) call the exit_loop tool or (2) rewrite the story.

Then we bring the agents together under a loop agent, which is itself nested inside of a sequential agent.

This design ensures that the system first produces an initial story draft, then the refinement loop runs up to the specified number of max_iterations.

You've now implemented a loop agent, creating a sophisticated system that can iteratively review and improve its own output. This is a key pattern for ensuring high-quality results.

You now have a complete toolkit of workflow patterns

In [61]:
# This Agent runs once at the beginning to create the Story Draft:
initial_writer_agent = Agent(
    name = 'InitialWriterAgent' ,
    model= Gemini(
        model="gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction= """ 
        Based on the user prompt write the first draft of a short story around 100-150 words. 
        Output only story text without an introduction or explanation.  """ ,
    output_key= "current_story"
)
print("Initial Writer Agent Created!")

#This Agent's job is to provide feedback or approval signal. It does not use any tools:
critic_agent = Agent(
    name = 'CriticAgent',
    model=Gemini(
        model = "gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction= """ 
        You are a constructive story critic. Review the story provided below.
        Story: {current_story}
    
        Evaluate the story's plot, characters, and pacing.
        - If the story is well-written and complete, and has a moral then you MUST respond with the exact phrase: "APPROVED"
        - Otherwise, provide 2-3 specific, actionable suggestions for improvement.
        """ ,
    output_key="critique"
)

print("critic_agent is created!")


Initial Writer Agent Created!
critic_agent is created!


In [62]:
# This is the function that the RefinerAgent will call to exit the loop.
def exit_loop():
    """Call this function ONLY when the critique is 'APPROVED', indicating the story is finished and no more changes are needed."""
    return {"status": "approved", "message": "Story approved. Exiting refinement loop."}


print("✅ exit_loop function created.")

# This agent refines the story based on critique OR calls the exit_loop function.
refiner_agent = Agent(
    name="RefinerAgent",
    model=Gemini(
        model="gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a story refiner. You have a story draft and critique.
    
    Story Draft: {current_story}
    Critique: {critique}
    
    Your task is to analyze the critique.
    - IF the critique is EXACTLY "APPROVED", identify the moral of the story and add it to the story then you MUST call the `exit_loop` function and nothing else.
    - OTHERWISE, rewrite the story draft to fully incorporate the feedback from the critique.""",
    output_key="current_story",  # It overwrites the story with the new, refined version.
    tools=[
        FunctionTool(exit_loop)
    ],  # The tool is now correctly initialized with the function reference.
)

print("✅ refiner_agent created.")

✅ exit_loop function created.
✅ refiner_agent created.


In [63]:
# The LoopAgent contains the agents that will run repeatedly: Critic -> Refiner.
story_refinement_loop = LoopAgent(
    name="StoryRefinementLoop",
    sub_agents=[critic_agent, refiner_agent],
    max_iterations=2,  # Prevents infinite loops
)

# The root agent is a SequentialAgent that defines the overall workflow: Initial Write -> Refinement Loop.
root_agent = SequentialAgent(
    name="StoryPipeline",
    sub_agents=[initial_writer_agent, story_refinement_loop],
)

print("✅ Loop and Sequential Agents created.")

✅ Loop and Sequential Agents created.


C:\Users\devas\AppData\Local\Temp\ipykernel_31228\1839283766.py:2: DeprecationWarning: LoopAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  story_refinement_loop = LoopAgent(
C:\Users\devas\AppData\Local\Temp\ipykernel_31228\1839283766.py:9: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  root_agent = SequentialAgent(


In [64]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Write a short story about a train driver who drove trains for over 20 years"
)

InitialWriterAgent > For twenty-three years, Arthur’s world was framed by a windshield and a ribbon of steel. He knew every curve of the Pennine route, every sleepy whistle-stop, and how the morning mist clung to the tracks near the old viaduct. 

Today was his final run. As he pulled the lever of the diesel engine, the familiar hum vibrated through his bones. Passengers bustled on the platform, unaware they were riding alongside history. Arthur looked out at the fading autumn light, feeling a profound mixture of sorrow and peace. 

He didn't need the speedometer; he felt the train's rhythm in his pulse. When they reached the final terminus, he cut the engine. The sudden silence was deafening. Arthur stood up, adjusted his cap one last time, and stepped out into a future without rails.
CriticAgent > APPROVED
RefinerAgent > For twenty-three years, Arthur’s world was framed by a windshield and a ribbon of steel. He knew every curve of the Pennine route, every sleepy whistle-stop, and how

### Implementing the above with Workflow()

The primary change is the check_critique(ctx) function:
- In LoopAgent: The agent had to manually call a tool function (exit_loop) to break the loop.
- In Workflow: The graph router (check_critique) inspects the critique directly. If it sees "APPROVED" (or reaches 2 iterations), it automatically diverts the flow to finish and exits. You no longer need the exit_loop function.

In addition there is a change in refiner_agent:
- refiner_agent works almost as-is, with two simple changes:
    - Required: Add mode="single_turn" (otherwise ADK throws the ValueError you saw earlier).
    - Optional (cleaner): Remove the tools=[FunctionTool(exit_loop)] and the "APPROVED" branch from its prompt, because refiner_agent is now only called when revisions are actually needed.

In [65]:
# This Agent runs once at the beginning to create the Story Draft:
initial_writer_agent = Agent(
    name = 'InitialWriterAgent' ,
    model= Gemini(
        model="gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction= """ 
        Based on the user prompt write the first draft of a short story around 100-150 words. 
        Output only story text without an introduction or explanation.  """ ,
    output_key= "current_story",   
)
print("Initial Writer Agent Created!")

#This Agent's job is to provide feedback or approval signal. It does not use any tools:
critic_agent = Agent(
    name = 'CriticAgent',
    model=Gemini(
        model = "gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction= """ 
        You are a constructive story critic. Review the story provided below.
        Story: {current_story}
    
        Evaluate the story's plot, characters, and pacing.
        - If the story is well-written and complete, and has a moral then you MUST respond with the exact phrase: "APPROVED"
        - Otherwise, provide 2-3 specific, actionable suggestions for improvement.
        """ ,
    output_key="critique",
    mode="single_turn"
)

print("critic_agent is created!")

from google.adk.workflow import Workflow, START, node
from google.adk.events import Event

refiner_agent = Agent(
    name="RefinerAgent",
    model=Gemini(
        model="gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a story refiner. You have a story draft and a critique.
    
    Story Draft: {current_story}
    Critique: {critique}
    
    Rewrite and improve the story draft to fully incorporate the suggestions from the critique.""",
    output_key="current_story",
    mode="single_turn",  # <-- Required for Workflow
)


# Guard node: checks critique & tracks loop iterations to prevent infinite cycles
@node(name="check_critique")
def check_critique(ctx):
    count = ctx.session.state.get("loop_count", 0)
    ctx.session.state["loop_count"] = count + 1
    
    critique = ctx.session.state.get("critique", "")
    
    # Exit loop if approved OR if max_iterations (2) reached
    if "APPROVED" in critique or count >= 2:
        return Event(route="exit")
    return Event(route="refine")

@node(name="finish")
def finish():
    return "Story refinement complete."

root_agent = Workflow(
    name="StoryPipeline",
    edges=[
        # Initial write -> critique -> condition check
        (START, initial_writer_agent, critic_agent, check_critique),
        
        # Route depending on check_critique output:
        (check_critique, {"refine": refiner_agent, "exit": finish}),
        
        # Refiner loops back to critic for review
        (refiner_agent, critic_agent),
    ]
)

print("✅ Loop Workflow created successfully.")


Initial Writer Agent Created!
critic_agent is created!
✅ Loop Workflow created successfully.


In [68]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Write an intentionally unfinished, broken draft about an alien: only 25 words, cutting off mid-sentence, with no ending and no moral."
)


InitialWriterAgent > Vessel hull compromised. Atmosphere venting. Subject seven-zero-four approaches the airlock carrying the strange, pulsing artifact. Its compound eyes reflect the dying console’s red-
CriticAgent > Here are 2-3 specific, actionable suggestions for improvement:

1. **Complete the Scene:** The story cuts off abruptly mid-word ("red-"). Ensure the narrative reaches a natural conclusion rather than stopping as a sentence fragment.
2. **Expand the Narrative Arc:** While the snippet sets up a high-stakes cliffhanger, it functions more as a hook than a complete story. Add a beginning and an ending to give the micro-fiction a sense of resolution or a complete thematic arc.
3. **Clarify the Stakes:** Give the reader a brief glimpse into *why* Subject seven-zero-four is carrying the artifact or what the consequences of the hull breach are, adding emotional or narrative weight to the moment.
RefinerAgent > Vessel hull compromised. Atmosphere venting. Subject seven-zero-four ap

## Section 6: Summary - Choosing the Right Pattern[¶](https://www.kaggle.com/code/kaggle5daysofai/day-1b-agent-architectures#Section-6:-Summary---Choosing-the-Right-Pattern)

### Decision Tree: Which Workflow Pattern?

![Agent Decision Tree](https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/agent-decision-tree.png)

### Quick Reference Table


| Pattern                     | When to Use                      | Example                    | Key Feature              |
| --------------------------- | -------------------------------- | -------------------------- | ------------------------ |
| **LLM-based (sub\_agents)** | Dynamic orchestration needed     | Research + Summarize       | LLM decides what to call |
| **Sequential**              | Order matters, linear pipeline   | Outline → Write → Edit   | Deterministic order      |
| **Parallel**                | Independent tasks, speed matters | Multi-topic research       | Concurrent execution     |
| **Loop**                    | Iterative improvement needed     | Writer + Critic refinement | Repeated cycles          |

---

## ✅ Congratulations! You're Now an Agent Orchestrator

In this notebook, you made the leap from a single agent to a **multi-agent system**.
You saw **why** a team of specialists is easier to build and debug than one "do-it-all" agent. Most importantly, you learned how to be the **director** of that team.

You used `SequentialAgent`, `ParallelAgent`, and `LoopAgent` to create deterministic workflows, and you even used an LLM as a 'manager' to make dynamic decisions. You also mastered the "plumbing" by using `output_key` to pass state between agents and make them collaborative.

### 📚 Learn More

Refer to the following documentation to learn more:

* [Agents in ADK](https://google.github.io/adk-docs/agents/)
* [Sequential Agents in ADK](https://google.github.io/adk-docs/agents/workflow-agents/sequential-agents/)
* [Parallel Agents in ADK](https://google.github.io/adk-docs/agents/workflow-agents/parallel-agents/)
* [Loop Agents in ADK](https://google.github.io/adk-docs/agents/workflow-agents/loop-agents/)
* [Custom Agents in ADK](https://google.github.io/adk-docs/agents/custom-agents/)
